# PaniPuri - Steel Pan Synthesizer

Play a calibrated steel pan synthesizer right in your browser. This notebook clones the PaniPuri repo and generates steel pan tones using parameters extracted from real Double Seconds pan recordings.

**No samples needed** — all sounds are synthesized from calibration data.

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/profLewis/paniPuri.git
%cd paniPuri
!pip install -q numpy scipy mido

In [ ]:
import numpy as np
from IPython.display import Audio, display, HTML
from synth import generate_note, load_calibration, get_note_params, midi_to_freq

SAMPLE_RATE = 44100
calibration = load_calibration('calibration.json')

def play_note(name, octave, duration=2.0):
    """Generate and play a single steel pan note."""
    NOTE_TO_SEMI = {'C':0,'C#':1,'Db':1,'D':2,'D#':3,'Eb':3,'E':4,'F':5,
                    'F#':6,'Gb':6,'G':7,'G#':8,'Ab':8,'A':9,'A#':10,'Bb':10,'B':11}
    midi = (octave + 1) * 12 + NOTE_TO_SEMI[name]
    freq = midi_to_freq(midi)
    params = get_note_params(calibration, midi)
    audio = generate_note(freq, params, duration=duration)
    display(HTML(f"<b>{name}{octave}</b> (MIDI {midi}, {freq:.1f} Hz)"))
    return Audio(audio, rate=SAMPLE_RATE, autoplay=True)

def play_chord(notes, duration=2.5):
    """Play multiple notes simultaneously. notes = [(name, octave), ...]"""
    NOTE_TO_SEMI = {'C':0,'C#':1,'Db':1,'D':2,'D#':3,'Eb':3,'E':4,'F':5,
                    'F#':6,'Gb':6,'G':7,'G#':8,'Ab':8,'A':9,'A#':10,'Bb':10,'B':11}
    mixed = np.zeros(int(SAMPLE_RATE * duration))
    names = []
    for name, octave in notes:
        midi = (octave + 1) * 12 + NOTE_TO_SEMI[name]
        freq = midi_to_freq(midi)
        params = get_note_params(calibration, midi)
        audio = generate_note(freq, params, duration=duration)
        mixed[:len(audio)] += audio
        names.append(f"{name}{octave}")
    mixed = np.tanh(mixed) * 0.85
    display(HTML(f"<b>Chord: {', '.join(names)}</b>"))
    return Audio(mixed, rate=SAMPLE_RATE, autoplay=True)

## Play Individual Notes

Try different notes across the tenor pan range (C4 to E6):

In [ ]:
play_note('C', 5)

In [ ]:
play_note('G', 5)

In [ ]:
play_note('E', 6)

## Play Chords

In [ ]:
# C major chord
play_chord([('C', 5), ('E', 5), ('G', 5)])

In [ ]:
# G major chord
play_chord([('G', 4), ('B', 4), ('D', 5)])

## Play a Melody

Generate a short steel pan melody and play it back:

In [ ]:
import time

def play_melody(notes, tempo=120, note_duration=0.5):
    """Play a sequence of notes. notes = [(name, octave), ...] or 'R' for rest."""
    NOTE_TO_SEMI = {'C':0,'C#':1,'Db':1,'D':2,'D#':3,'Eb':3,'E':4,'F':5,
                    'F#':6,'Gb':6,'G':7,'G#':8,'Ab':8,'A':9,'A#':10,'Bb':10,'B':11}
    beat_dur = 60.0 / tempo
    total_samples = int(SAMPLE_RATE * len(notes) * beat_dur * note_duration) + SAMPLE_RATE * 2
    mixed = np.zeros(total_samples)
    pos = 0
    step = int(SAMPLE_RATE * beat_dur * note_duration)
    for note in notes:
        if note == 'R' or note is None:
            pos += step
            continue
        name, octave = note
        midi = (octave + 1) * 12 + NOTE_TO_SEMI[name]
        freq = midi_to_freq(midi)
        params = get_note_params(calibration, midi)
        audio = generate_note(freq, params, duration=1.5)
        end = min(pos + len(audio), total_samples)
        mixed[pos:end] += audio[:end-pos]
        pos += step
    mixed = np.tanh(mixed) * 0.85
    return Audio(mixed, rate=SAMPLE_RATE, autoplay=True)

# "Mary Had a Little Lamb" on steel pan
melody = [
    ('E', 5), ('D', 5), ('C', 5), ('D', 5),
    ('E', 5), ('E', 5), ('E', 5), 'R',
    ('D', 5), ('D', 5), ('D', 5), 'R',
    ('E', 5), ('G', 5), ('G', 5), 'R',
    ('E', 5), ('D', 5), ('C', 5), ('D', 5),
    ('E', 5), ('E', 5), ('E', 5), ('E', 5),
    ('D', 5), ('D', 5), ('E', 5), ('D', 5),
    ('C', 5), 'R', 'R', 'R',
]

play_melody(melody, tempo=140)

## All 29 Tenor Pan Notes

Generate a chromatic scale across the full range:

In [ ]:
NOTE_NAMES = ['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']

# Full tenor pan range: C4 to E6
all_notes = []
for midi in range(60, 89):  # C4 to E6
    name = NOTE_NAMES[midi % 12]
    octave = (midi // 12) - 1
    all_notes.append((name, octave))

play_melody(all_notes, tempo=240, note_duration=0.5)